# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates loading and exploring the FAIR^2 colorectal cancer survivors dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source

The dataset is described and made accessible via a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load the dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is an object, not dict

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Date Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets (tables), their fields (columns), and entity `@id`s. All references will use the Croissant schema `@id` fields for rigor and clarity.

In [ ]:
# List record sets available in the dataset, referencing each by its @id

record_sets = dataset.record_sets  # a dict mapping @id to RecordSet objects

print("Available record sets:")
for rs_id, rs in record_sets.items():
    print(f"  - @id: {rs_id}")
    print(f"      name: {rs.name}")
    print(f"      description: {rs.description}")
    # List fields for this record set
    print(f"      Fields / columns:")
    for field_id, field in rs.fields.items():
        print(f"        - @id: {field_id}")
        print(f"            name: {field.name}")
            # print(f"            dataType: {getattr(field, 'dataType', None)}")
    print("---")

## 3. Data Extraction
Load the data from specific record set(s) into pandas DataFrame(s) for further analysis.

Below, all names are referenced by their `@id`.

In [ ]:
# Collect all record set @ids
record_set_ids = list(record_sets.keys())
print("Record set @ids:")
print(record_set_ids)

# Load records for each record set as DataFrames, referenced by @id
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading for record set {rs_id} ...")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Display columns of the first record set as an example
if record_set_ids:
    example_rs_id = record_set_ids[0]
    print(f"Columns in {example_rs_id}:")
    print(dataframes[example_rs_id].columns.tolist())
    display(dataframes[example_rs_id].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)

Apply typical EDA steps, using only `@id` references for all fields.

Let's:
- Select a numeric field (e.g., 'age')
- Filter records with age > 60
- Normalize the selected field
- Optionally group by a categorical variable (e.g., 'Sex')

You can adapt this pattern for other fields as desired.

*If you do not know which fields are numeric, please check the record set overview above, or apply* `dataframes[<record_set_id>].describe()` *to inspect the dataframe.*

In [ ]:
# Pick the first record set for demonstration and show summary stats
if record_set_ids:
    rs_id = example_rs_id  # use the example record set ID
    df = dataframes[rs_id]
    print(f"Record set: {rs_id}")
    print("Columns:", df.columns.tolist())
    display(df.describe(include='all'))
    # Try to find a numeric column by type or name
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_candidates:
        # Try common names
        for col in df.columns:
            if 'age' in col.lower() or 'interval' in col.lower() or 'n' in col.lower():
                numeric_candidates.append(col)
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = 60
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())
        # Normalize the numeric field
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())
        # Group by a categorical field (e.g., by 'Sex' or another field, using @id as column name)
        group_field_candidates = [col for col in df.columns if col.lower() in ['sex', 'gender', 'msi_status', 'anatomical_location']]
        if group_field_candidates:
            group_field = group_field_candidates[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame().reset_index()
            print(f"Mean {numeric_field} grouped by {group_field}:")
            display(grouped_df)
        else:
            print("No suitable group field found in columns.")
    else:
        print("No obvious numeric field found.")
else:
    print("No record sets available for EDA.")

## 5. Visualization

Visualize the distribution of the selected numeric field and illustrate a categorical breakdown if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only proceed if a numeric field and filtered data are available
if 'filtered_df' in locals() and 'numeric_field' in locals() and not filtered_df.empty:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field], kde=True, bins=20)
    plt.title(f'Histogram of {numeric_field} (filtered > {threshold})')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group field if available
    if 'group_field' in locals():
        plt.figure(figsize=(7, 5))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("Not enough filtered numeric data to plot.")

## 6. Conclusion

- This notebook demonstrated how to load FAIR^2 colorectal cancer survivors data via Croissant schema and the `mlcroissant` library, referencing all dataset components via their `@id`s.
- You performed EDA by selecting and normalizing numeric variables, filtering data, and visualizing key field distributions.
- You may use this pattern to examine other fields or record sets, adapting the code for more domain-specific insights.